In [ ]:
import pandas as pd
import requests
import keyring
import urllib3
from pyodata import Client
from datetime import datetime

# =================================================================
# 1. 基础配置与安全设置
# =================================================================
SERVICE_URL = 'https://yousap/sap/opu/odata/sap/ZMTEST5_UMD_SRV/'
SAP_USER = "sap userid"
SERVICE_NAME = "sap_100"

# 关闭 SSL 警告（针对 SAP 自签名证书）
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def generate_sap_report():
    try:
        print("正在连接 SAP 并获取数据...")
        
        # 2. 安全获取密码并建立会话
        pwd = keyring.get_password(SERVICE_NAME, SAP_USER)
        if not pwd:
            raise ValueError(f"在 keyring 中未找到服务 {SERVICE_NAME} 的密码，请先设置密码。")

        session = requests.Session()
        session.auth = (SAP_USER, pwd)
        session.verify = False
        session.timeout = 15

        # 3. 初始化 pyodata 客户端并抓取数据
        sap_client = Client(SERVICE_URL, session)
        # 获取所有航司实体
        entities = sap_client.entity_sets.scarrSet.get_entities().execute()
        
        # 4. 转换数据为 DataFrame 方便处理
        data = []
        for entity in entities:
            data.append({
                "ID": entity.Carrid,
                "Name": entity.Carrname,
                "Currency": entity.Currcode,
                "Url": entity.Url if entity.Url else "#"
            })
        df = pd.DataFrame(data)

        # =================================================================
        # 5. 构建极简奢华版 HTML 模板 (内联 CSS)
        # =================================================================
        now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        html_content = f"""
        <!DOCTYPE html>
        <html lang="zh-CN">
        <head>
            <meta charset="UTF-8">
            <style>
                :root {{
                    --sap-blue: #0070d2;
                    --text-main: #32363a;
                    --text-sub: #6a6d70;
                    --bg-light: #f3f4f5;
                    --white: #ffffff;
                }}
                
                body {{ 
                    background-color: var(--bg-light); 
                    font-family: 'Segoe UI', 'Microsoft YaHei', sans-serif; 
                    margin: 0; padding: 40px 20px;
                    color: var(--text-main);
                }}

                .report-card {{
                    max-width: 1000px;
                    margin: 0 auto;
                    background: var(--white);
                    border-radius: 12px;
                    box-shadow: 0 8px 24px rgba(0,0,0,0.08);
                    overflow: hidden;
                }}

                .header {{
                    padding: 30px;
                    background: linear-gradient(135deg, #004487 0%, #0070d2 100%);
                    color: white;
                }}

                .header h1 {{ margin: 0; font-size: 24px; font-weight: 500; letter-spacing: 1px; }}
                .header .info {{ margin-top: 10px; font-size: 13px; opacity: 0.8; line-height: 1.6; }}

                .content {{ padding: 20px; }}

                table {{ 
                    width: 100%; border-collapse: collapse; margin-top: 10px; 
                }}
                
                th {{ 
                    text-align: left; padding: 16px; 
                    background-color: #fafafa;
                    color: var(--text-sub); font-weight: 600;
                    font-size: 14px; border-bottom: 2px solid #eeeeee;
                }}

                td {{ 
                    padding: 16px; border-bottom: 1px solid #f0f0f0; 
                    font-size: 14px; vertical-align: middle;
                }}

                tr:last-child td {{ border-bottom: none; }}
                tr:hover {{ background-color: #f8fbff; }}

                /* 样式组件 */
                .id-badge {{
                    background: #e1f0ff; color: #0056b3;
                    padding: 4px 10px; border-radius: 6px;
                    font-family: 'Consolas', monospace; font-weight: bold;
                }}

                .curr-tag {{
                    background: #f0f0f0; color: #555;
                    padding: 2px 8px; border-radius: 4px; font-size: 12px;
                }}

                .btn-link {{
                    display: inline-block;
                    text-decoration: none; color: white;
                    background: var(--sap-blue);
                    padding: 6px 16px; border-radius: 20px;
                    font-size: 12px; transition: all 0.3s;
                }}

                .btn-link:hover {{ background: #005fb2; transform: translateY(-1px); box-shadow: 0 4px 8px rgba(0,0,0,0.15); }}
                
                .footer {{
                    text-align: center; padding: 20px; 
                    font-size: 12px; color: var(--text-sub);
                    background: #fafafa; border-top: 1px solid #eee;
                }}
            </style>
        </head>
        <body>
            <div class="report-card">
                <div class="header">
                    <h1>✈️ 航司基础数据中心</h1>
                    <div class="info">
                        <strong>接口地址:</strong> {SERVICE_URL}<br>
                        <strong>提取时间:</strong> {now_str} | <strong>用户:</strong> {SAP_USER}
                    </div>
                </div>
                
                <div class="content">
                    <table>
                        <thead>
                            <tr>
                                <th width="15%">航司代码</th>
                                <th width="45%">航空公司全称</th>
                                <th width="15%">法定币种</th>
                                <th width="25%">官方入口</th>
                            </tr>
                        </thead>
                        <tbody>
        """

        # 循环填充数据行
        for _, row in df.iterrows():
            html_content += f"""
                            <tr>
                                <td><span class="id-badge">{row['ID']}</span></td>
                                <td style="font-weight: 500;">{row['Name']}</td>
                                <td><span class="curr-tag">{row['Currency']}</span></td>
                                <td><a href="{row['Url']}" target="_blank" class="btn-link">查看详情</a></td>
                            </tr>
            """

        html_content += f"""
                        </tbody>
                    </table>
                </div>
                <div class="footer">
                    © {datetime.now().year} 数字化供应链报表系统 - 自动生成
                </div>
            </div>
        </body>
        </html>
        """

        # 6. 保存报表文件
        filename = "SAP_Airline_Report.html"
        with open(filename, "w", encoding="utf-8") as f:
            f.write(html_content)
        
        print(f"✨ 报表生成成功！请查看当前目录下的: {filename}")

    except Exception as e:
        print(f"❌ 运行失败: {e}")

if __name__ == "__main__":
    generate_sap_report()
